# Generador 3D Multi-Vista Gratuito (Hunyuan3D-2mv) -> STL (Peana 30x30 mm)
Este notebook se ejecuta en los servidores de Google Colab (fuera de Zscaler), descarga `elektra_frontal.png` y `elektra_trasera.png` de tu repositorio `inigoku/mu`, genera el modelo 3D completo mediante **Tencent Hunyuan3D-2mv (Multi-Vista)** de forma 100% gratuita y descarga el archivo `.STL` con peana de **30x30 mm**.

In [ ]:
!pip install -q gradio_client trimesh numpy scipy pillow
!git clone https://github.com/inigoku/mu.git /content/mu || (cd /content/mu && git pull)

In [ ]:
from pathlib import Path
import numpy as np
import trimesh
from gradio_client import Client, handle_file
from google.colab import files

front_png = '/content/mu/elektra_frontal.png'
back_png = '/content/mu/elektra_trasera.png'
output_stl = '/content/elektra_hunyuan3d_30x30.stl'

print('Conectando con tencent/Hunyuan3D-2mv (Multi-Vista Frontal + Trasera)...')
client = Client('tencent/Hunyuan3D-2mv')
result = client.predict(
    image=handle_file(front_png),
    mv_image_front=handle_file(front_png),
    mv_image_back=handle_file(back_png),
    mv_image_left=None,
    mv_image_right=None,
    steps=30,
    guidance_scale=5.0,
    seed=1234,
    octree_resolution=256,
    check_box_rembg=False,
    num_chunks=8000,
    randomize_seed=False,
    api_name='/shape_generation'
)

mesh_path = result[0] if isinstance(result, (list, tuple)) else result
if isinstance(mesh_path, dict) and 'value' in mesh_path:
    mesh_path = mesh_path['value']

mesh = trimesh.load(mesh_path, force='mesh')
v = mesh.vertices.copy()
if mesh.extents[1] > mesh.extents[2]:
    mesh.vertices = np.column_stack([v[:, 0], -v[:, 2], v[:, 1]])

mesh.vertices[:, 2] -= mesh.bounds[0, 2]
total_h = mesh.extents[2]
base_verts = mesh.vertices[mesh.vertices[:, 2] <= 0.12 * total_h]
base_w = float(base_verts[:, 0].max() - base_verts[:, 0].min())
base_d = float(base_verts[:, 1].max() - base_verts[:, 1].min())

sx = 30.0 / max(base_w, 1e-5)
sy = 30.0 / max(base_d, 1e-5)
sz = sx
cx = 0.5 * float(base_verts[:, 0].max() + base_verts[:, 0].min())
cy = 0.5 * float(base_verts[:, 1].max() + base_verts[:, 1].min())

mesh.vertices[:, 0] = (mesh.vertices[:, 0] - cx) * sx
mesh.vertices[:, 1] = (mesh.vertices[:, 1] - cy) * sy
mesh.vertices[:, 2] *= sz

mesh.export(output_stl)
print(f'STL generado: {output_stl} | Dimensiones totales: {mesh.extents.round(1)} mm')
files.download(output_stl)